In [24]:
# =====================================================
# FULL PHASE 1 - GEOCODER.CA WITH AUTHENTICATION CODE
# =====================================================

import pandas as pd
import re
import requests
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_colwidth', 200)

# ------------------- PUT YOUR GEOCODER.CA AUTH CODE HERE -------------------
AUTH_CODE = "43291987525357180330x214751088"   # ←←← Paste your authentication code

# ------------------- LOAD RAW FILES -------------------
base_path = Path(r'C:\Users\MSI\Desktop\ResourceLink Discovery\Data')

mh_search = pd.read_csv(base_path / 'ontario_centers_deduplicated_mh.csv')
at_search = pd.read_csv(base_path / 'Addiction_treatment_results.csv')
mh_gt = pd.read_csv(base_path / '211_Community Mental Health Centres_resources_GT.csv')
at_gt = pd.read_csv(base_path / '211_Addiction Treatment_resources_GT.csv')

print("✅ Raw files loaded.")

# ------------------- CLEANING FUNCTIONS -------------------
def clean_service_name(name):
    if pd.isna(name) or str(name).strip() == "":
        return ""
    name = str(name).lower().strip()
    name = re.sub(r'\s*\([^)]*\)', '', name)
    name = re.sub(r'[-–—]', ' ', name)
    name = re.sub(r'[^\w\s&]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

def extract_city(addr):
    if pd.isna(addr) or str(addr).strip() == "":
        return ""
    addr = str(addr).strip()
    addr_lower = addr.lower()
    for city in sorted(ontario_cities, key=len, reverse=True):
        if city.lower() in addr_lower:
            return city
    if ',' in addr:
        parts = [p.strip() for p in addr.split(',')]
        for i in range(len(parts)-1, -1, -1):
            candidate = re.sub(r'\s*(ON|Ontario|Canada).*?$', '', parts[i], flags=re.I).strip()
            if len(candidate) > 4 and not any(street in candidate.lower() for street in ['st ', 'street', 'ave ', 'rd ', 'main st', 'king st', 'dundas', 'cannon']):
                return candidate.title()
    match = re.search(r'([A-Za-z\s\.\'-]{5,})', addr)
    if match:
        candidate = match.group(1).strip().title()
        if len(candidate) > 4:
            return candidate
    return ""

ontario_cities = {"Toronto", "Ottawa", "Hamilton", "Mississauga", "Brampton", "London", "Markham", "Vaughan", "Kitchener", "Windsor", "Richmond Hill", "Oakville", "Burlington", "Greater Sudbury", "Sudbury", "Oshawa", "St. Catharines", "St Catharines", "Barrie", "Guelph", "Cambridge", "Whitby", "Ajax", "Pickering", "Brantford", "Newmarket", "Milton", "Peterborough", "Kingston", "Thunder Bay", "Waterloo", "Sarnia", "Aurora", "Welland", "North Bay", "Belleville", "Cornwall", "Chatham-Kent", "Chatham", "Niagara Falls", "Orangeville", "Woodstock", "Orillia", "Timmins", "Quinte West", "East Gwillimbury", "Georgina", "Brockville", "Stratford", "Sault Ste Marie", "Sault Ste. Marie", "Owen Sound", "Grimsby", "Innisfil", "Collingwood", "Bradford West Gwillimbury", "Fort Erie", "Port Colborne", "Thorold", "St Thomas", "Leamington", "Amherstburg", "Tecumseh", "Penetanguishene", "Midland", "Wasaga Beach", "Bracebridge", "Gravenhurst", "Huntsville", "Parry Sound", "Hawkesbury", "Merrickville", "Merrickville-Wolford"}

# Apply cleaning
for df in [mh_search, at_search, mh_gt, at_gt]:
    col = 'Service name' if 'Service name' in df.columns else 'Name'
    df['Clean_Service_Name'] = df[col].apply(clean_service_name)

mh_search['City'] = mh_search['address'].apply(extract_city)
at_search['City'] = at_search['address'].apply(extract_city)

# ------------------- GEOCODER.CA WITH AUTH CODE -------------------
postal_cache = {}
def get_city_from_postal_geocoder(postal_code):
    if pd.isna(postal_code) or str(postal_code).strip() == "":
        return None
    postal = str(postal_code).strip().replace(" ", "").upper()
    if len(postal) not in [6, 7]:
        return None
    if postal in postal_cache:
        return postal_cache[postal]
    try:
        url = f"https://geocoder.ca/{postal}?json=1&auth={AUTH_CODE}"
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            city = data.get('standard', {}).get('city')
            if city:
                postal_cache[postal] = city
                return city
    except:
        pass
    postal_cache[postal] = None
    return None

print("Applying Geocoder.ca with authentication code...")
api_calls = 0
api_fixed = 0
for df in [mh_search, at_search]:
    mask = (df['City'] == "") & df['Postal code'].notna()
    for idx, row in df[mask].iterrows():
        city = get_city_from_postal_geocoder(row['Postal code'])
        api_calls += 1
        if city:
            df.at[idx, 'City'] = city
            api_fixed += 1

print(f"Geocoder.ca calls made : {api_calls} | Successfully fixed : {api_fixed} cities")

# ------------------- LABELING -------------------
for df in [mh_search, at_search]:
    df['City_Extraction_Status'] = "Good"
    df.loc[df['address'].isna() | (df['address'].str.strip() == ""), 'City_Extraction_Status'] = "Empty Address"
    df.loc[df['City'] == "", 'City_Extraction_Status'] = "Empty City"
    df.loc[(df['City'].str.len() > 0) & (df['City'].str.len() < 5), 'City_Extraction_Status'] = "Short City"

# ------------------- SAVE + CATEGORIZED FILES -------------------
output_dir = Path('phase1_output')
output_dir.mkdir(exist_ok=True)

mh_search.to_csv(output_dir / 'mh_search_cleaned.csv', index=False)
at_search.to_csv(output_dir / 'at_search_cleaned.csv', index=False)
mh_gt.to_csv(output_dir / 'mh_gt_cleaned.csv', index=False)
at_gt.to_csv(output_dir / 'at_gt_cleaned.csv', index=False)

for name, df in [("mh", mh_search), ("at", at_search)]:
    df[df['City_Extraction_Status'] == "Good"].to_csv(output_dir / f'good_{name}.csv', index=False)
    df[df['City_Extraction_Status'] == "Empty City"].to_csv(output_dir / f'empty_city_{name}.csv', index=False)
    df[df['City_Extraction_Status'] == "Empty Address"].to_csv(output_dir / f'empty_address_{name}.csv', index=False)
    df[df['Postal code'].isna() | (df['Postal code'].astype(str).str.strip() == "")].to_csv(output_dir / f'no_postal_code_{name}.csv', index=False)
    df[(df['City'].isna() | (df['City'].astype(str).str.strip() == "")) & 
       (df['address'].isna() | (df['address'].astype(str).str.strip() == "")) & 
       (df['Postal code'].isna() | (df['Postal code'].astype(str).str.strip() == ""))].to_csv(output_dir / f'completely_missing_{name}.csv', index=False)


✅ Raw files loaded.
Applying Geocoder.ca with authentication code...
Geocoder.ca calls made : 269 | Successfully fixed : 257 cities


In [25]:
# =====================================================
# FULL NUMERICAL DISTRIBUTION REPORT (CURRENT FILES)
# =====================================================

import pandas as pd
from pathlib import Path

output_dir = Path('phase1_output')

mh = pd.read_csv(output_dir / 'mh_search_cleaned.csv')
at = pd.read_csv(output_dir / 'at_search_cleaned.csv')

print("=== FULL DISTRIBUTION REPORT - ALL CASES ===\n")

for name, df in [("Mental Health", mh), ("Addiction Treatment", at)]:
    total = len(df)
    good = (df['City_Extraction_Status'] == "Good").sum()
    empty_city = (df['City_Extraction_Status'] == "Empty City").sum()
    empty_addr = (df['City_Extraction_Status'] == "Empty Address").sum()
    short_city = (df['City_Extraction_Status'] == "Short City").sum()
    suspicious = (df['City_Extraction_Status'] == "Suspicious").sum()
    no_postal = (df['Postal code'].isna() | (df['Postal code'].astype(str).str.strip() == "")).sum()
    
    completely_missing = len(df[
        (df['City'].isna() | (df['City'].astype(str).str.strip() == "")) &
        (df['address'].isna() | (df['address'].astype(str).str.strip() == "")) &
        (df['Postal code'].isna() | (df['Postal code'].astype(str).str.strip() == ""))
    ])
    
    print(f"{name}:")
    print(f"   Total rows                  : {total:,}")
    print(f"   ✅ Good                     : {good:,} ({good/total*100:.1f}%)")
    print(f"   Empty City                  : {empty_city:,} ({empty_city/total*100:.1f}%)")
    print(f"   Empty Address               : {empty_addr:,} ({empty_addr/total*100:.1f}%)")
    print(f"   Short City (<5 chars)       : {short_city:,} ({short_city/total*100:.1f}%)")
    print(f"   Suspicious                  : {suspicious:,} ({suspicious/total*100:.1f}%)")
    print(f"   No Postal Code              : {no_postal:,} ({no_postal/total*100:.1f}%)")
    print(f"   🔥 Completely Missing All 3 : {completely_missing:,} ({completely_missing/total*100:.1f}%)")
    print("-" * 70)

=== FULL DISTRIBUTION REPORT - ALL CASES ===

Mental Health:
   Total rows                  : 5,581
   ✅ Good                     : 3,480 (62.4%)
   Empty City                  : 2,065 (37.0%)
   Empty Address               : 32 (0.6%)
   Short City (<5 chars)       : 4 (0.1%)
   Suspicious                  : 0 (0.0%)
   No Postal Code              : 3,183 (57.0%)
   🔥 Completely Missing All 3 : 2,043 (36.6%)
----------------------------------------------------------------------
Addiction Treatment:
   Total rows                  : 5,742
   ✅ Good                     : 3,968 (69.1%)
   Empty City                  : 1,603 (27.9%)
   Empty Address               : 166 (2.9%)
   Short City (<5 chars)       : 5 (0.1%)
   Suspicious                  : 0 (0.0%)
   No Postal Code              : 3,096 (53.9%)
   🔥 Completely Missing All 3 : 1,587 (27.6%)
----------------------------------------------------------------------


In [23]:
import requests

postal = "L8S4L8"   # ← change this to one postal code from your suspicious file

url = f"https://geocoder.ca/{postal}?json=1"
r = requests.get(url, timeout=10)
print("Status code:", r.status_code)

if r.status_code == 200:
    data = r.json()
    city = data.get('standard', {}).get('city')
    print("City returned:", city)
else:
    print("Response:", r.text[:300])

Status code: 403
Response: {
  "success": false,
  "error": {
    "code": "006",
    "message": "Request Throttled. Visit Geocoder.ca to obtain an authentication code."
  }
}



In [26]:
# =====================================================
# FULL PHASE 1 - GEOCODER.CA WITH POSTAL + ADDRESS FALLBACK
# =====================================================

import pandas as pd
import re
import requests
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_colwidth', 200)

# ------------------- YOUR GEOCODER.CA AUTH CODE -------------------
AUTH_CODE = "43291987525357180330x214751088"   # ← Paste your auth code here

# ------------------- LOAD RAW FILES -------------------
base_path = Path(r'C:\Users\MSI\Desktop\ResourceLink Discovery\Data')

mh_search = pd.read_csv(base_path / 'ontario_centers_deduplicated_mh.csv')
at_search = pd.read_csv(base_path / 'Addiction_treatment_results.csv')
mh_gt = pd.read_csv(base_path / '211_Community Mental Health Centres_resources_GT.csv')
at_gt = pd.read_csv(base_path / '211_Addiction Treatment_resources_GT.csv')

print("✅ Raw files loaded.")

# ------------------- CLEANING FUNCTIONS -------------------
def clean_service_name(name):
    if pd.isna(name) or str(name).strip() == "":
        return ""
    name = str(name).lower().strip()
    name = re.sub(r'\s*\([^)]*\)', '', name)
    name = re.sub(r'[-–—]', ' ', name)
    name = re.sub(r'[^\w\s&]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

def extract_city(addr):
    if pd.isna(addr) or str(addr).strip() == "":
        return ""
    addr = str(addr).strip()
    addr_lower = addr.lower()
    for city in sorted(ontario_cities, key=len, reverse=True):
        if city.lower() in addr_lower:
            return city
    if ',' in addr:
        parts = [p.strip() for p in addr.split(',')]
        for i in range(len(parts)-1, -1, -1):
            candidate = re.sub(r'\s*(ON|Ontario|Canada).*?$', '', parts[i], flags=re.I).strip()
            if len(candidate) > 4 and not any(street in candidate.lower() for street in ['st ', 'street', 'ave ', 'rd ', 'main st', 'king st', 'dundas', 'cannon']):
                return candidate.title()
    match = re.search(r'([A-Za-z\s\.\'-]{5,})', addr)
    if match:
        candidate = match.group(1).strip().title()
        if len(candidate) > 4:
            return candidate
    return ""

ontario_cities = {"Toronto", "Ottawa", "Hamilton", "Mississauga", "Brampton", "London", "Markham", "Vaughan", "Kitchener", "Windsor", "Richmond Hill", "Oakville", "Burlington", "Greater Sudbury", "Sudbury", "Oshawa", "St. Catharines", "St Catharines", "Barrie", "Guelph", "Cambridge", "Whitby", "Ajax", "Pickering", "Brantford", "Newmarket", "Milton", "Peterborough", "Kingston", "Thunder Bay", "Waterloo", "Sarnia", "Aurora", "Welland", "North Bay", "Belleville", "Cornwall", "Chatham-Kent", "Chatham", "Niagara Falls", "Orangeville", "Woodstock", "Orillia", "Timmins", "Quinte West", "East Gwillimbury", "Georgina", "Brockville", "Stratford", "Sault Ste Marie", "Sault Ste. Marie", "Owen Sound", "Grimsby", "Innisfil", "Collingwood", "Bradford West Gwillimbury", "Fort Erie", "Port Colborne", "Thorold", "St Thomas", "Leamington", "Amherstburg", "Tecumseh", "Penetanguishene", "Midland", "Wasaga Beach", "Bracebridge", "Gravenhurst", "Huntsville", "Parry Sound", "Hawkesbury", "Merrickville", "Merrickville-Wolford"}

# Apply cleaning
for df in [mh_search, at_search, mh_gt, at_gt]:
    col = 'Service name' if 'Service name' in df.columns else 'Name'
    df['Clean_Service_Name'] = df[col].apply(clean_service_name)

mh_search['City'] = mh_search['address'].apply(extract_city)
at_search['City'] = at_search['address'].apply(extract_city)

# ------------------- GEOCODER.CA WITH POSTAL + ADDRESS FALLBACK -------------------
postal_cache = {}
def get_city_from_address_or_postal(postal_code, address):
    """Try postal first, then full address as fallback"""
    # 1. Try postal code
    if pd.notna(postal_code) and str(postal_code).strip() != "":
        postal = str(postal_code).strip().replace(" ", "").upper()
        if postal in postal_cache:
            return postal_cache[postal]
        try:
            url = f"https://geocoder.ca/{postal}?json=1&auth={AUTH_CODE}"
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                city = r.json().get('standard', {}).get('city')
                if city:
                    postal_cache[postal] = city
                    return city
        except:
            pass

    # 2. Fallback: Try full address
    if pd.notna(address) and str(address).strip() != "":
        addr = str(address).strip().replace(" ", "+")
        try:
            url = f"https://geocoder.ca/?locate={addr}&json=1&auth={AUTH_CODE}"
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                city = r.json().get('standard', {}).get('city')
                if city:
                    return city
        except:
            pass
    return None

print("Applying Geocoder.ca (postal + address fallback)...")
api_calls = 0
api_fixed = 0

for df in [mh_search, at_search]:
    mask = (df['City'] == "")   # only rows still missing city
    for idx, row in df[mask].iterrows():
        city = get_city_from_address_or_postal(row.get('Postal code'), row.get('address'))
        api_calls += 1
        if city:
            df.at[idx, 'City'] = city
            api_fixed += 1

print(f"Geocoder.ca calls made : {api_calls} | Successfully fixed : {api_fixed} cities")

# ------------------- LABELING & SAVE -------------------
for df in [mh_search, at_search]:
    df['City_Extraction_Status'] = "Good"
    df.loc[df['address'].isna() | (df['address'].str.strip() == ""), 'City_Extraction_Status'] = "Empty Address"
    df.loc[df['City'] == "", 'City_Extraction_Status'] = "Empty City"
    df.loc[(df['City'].str.len() > 0) & (df['City'].str.len() < 5), 'City_Extraction_Status'] = "Short City"

output_dir = Path('phase1_output')
output_dir.mkdir(exist_ok=True)

mh_search.to_csv(output_dir / 'mh_search_cleaned.csv', index=False)
at_search.to_csv(output_dir / 'at_search_cleaned.csv', index=False)
mh_gt.to_csv(output_dir / 'mh_gt_cleaned.csv', index=False)
at_gt.to_csv(output_dir / 'at_gt_cleaned.csv', index=False)

print("\n✅ Phase 1 completed with address fallback!")

# Simple report
print("\n" + "="*80)
print("PHASE 1 FINAL REPORT")
print("="*80)
for name, df in [("Mental Health", mh_search), ("Addiction Treatment", at_search)]:
    total = len(df)
    good = (df['City_Extraction_Status'] == "Good").sum()
    empty_city = (df['City_Extraction_Status'] == "Empty City").sum()
    print(f"\n{name}:")
    print(f"   Total rows          : {total:,}")
    print(f"   ✅ Good             : {good:,} ({good/total*100:.1f}%)")
    print(f"   Empty City          : {empty_city:,} ({empty_city/total*100:.1f}%)")

✅ Raw files loaded.
Applying Geocoder.ca (postal + address fallback)...
Geocoder.ca calls made : 3925 | Successfully fixed : 270 cities

✅ Phase 1 completed with address fallback!

PHASE 1 FINAL REPORT

Mental Health:
   Total rows          : 5,581
   ✅ Good             : 3,481 (62.4%)
   Empty City          : 2,055 (36.8%)

Addiction Treatment:
   Total rows          : 5,742
   ✅ Good             : 3,968 (69.1%)
   Empty City          : 1,600 (27.9%)
